# Get hydro data

We'll use this data in our streamflow prediction model for East Canyon, UT.

Use the 'hyriver' environment.

In [13]:
from pynhd import NLDI
import geopandas as gpd
import pandas as pd
from supportingscripts import getData, SNOTEL_Analyzer, dataprocessing, mapping
from shapely.geometry import box, Polygon
import os
import datetime
import matplotlib.pyplot as plt
import numpy as np
import pydaymet as daymet
import warnings
warnings.filterwarnings("ignore")

In [4]:
station_id = "10133980" # NWIS id for East Canyon Creek above East Canyon Reservoir near Morgan, UT
basinname = 'EastCanyonCreek'

Load SNOTEL data that was already downloaded.

In [5]:
#load snotel data
unprocessed_SNOTEL = {}
#read all files in the following path into the dictionary
path = 'files/SNOTEL'
for filename in os.listdir(path):
    if filename.endswith('.csv'):
        #select the name of the file between the _ and _
        name = filename.split('_')[1] 
        unprocessed_SNOTEL[name] = pd.read_csv(os.path.join(path, filename))
        #make the date a datetime object and set to the index
        unprocessed_SNOTEL[name]['Date'] = pd.to_datetime(unprocessed_SNOTEL[name]['Date'])
        unprocessed_SNOTEL[name].set_index('Date', inplace=True)
        #rename the Snow Water Equivalent (m) Start of Day Values to SWE_cm
        unprocessed_SNOTEL[name].rename(columns={'Snow Water Equivalent (m) Start of Day Values': f"{name}_SWE_cm"}, inplace=True)
        #convert SWE_m to cm
        unprocessed_SNOTEL[name][f"{name}_SWE_cm"] = unprocessed_SNOTEL[name][f"{name}_SWE_cm"] * 100
        #remove the Water_Year column
        unprocessed_SNOTEL[name].drop(columns=['Water_Year'], inplace=True)
        #we need to know how many obs for each DF, print the df name, its length, and the start/end dates
        print(f"{name}: {len(unprocessed_SNOTEL[name])} start date: {unprocessed_SNOTEL[name].index.min()} end date: {unprocessed_SNOTEL[name].index.max()}")
    


814: 13795 start date: 1988-06-20 00:00:00 end date: 2026-03-27 00:00:00
684: 17345 start date: 1978-10-01 00:00:00 end date: 2026-03-27 00:00:00


In [6]:
#The site with the latest start date will guide the rest
latest_start_date = max([df.index.min() for df in unprocessed_SNOTEL.values()])

#The site with the earliest end date will guide the rest
soonest_end_date = min([df.index.max() for df in unprocessed_SNOTEL.values()])
for key in unprocessed_SNOTEL.keys():
    unprocessed_SNOTEL[key] = unprocessed_SNOTEL[key][unprocessed_SNOTEL[key].index >= latest_start_date]
    unprocessed_SNOTEL[key] = unprocessed_SNOTEL[key][unprocessed_SNOTEL[key].index <= soonest_end_date]

#merge all dictionary dataframes into one larger dataframe
SNOTEL_df = pd.concat(unprocessed_SNOTEL.values(), axis=1)
#set the date index to be the index of the first dataframe in the dictionary

SNOTEL_df.head()

,814_SWE_cm,684_SWE_cm
Date,,
1988-06-20,0.0,0.0
1988-06-21,0.0,0.0
1988-06-22,0.0,0.0
1988-06-23,0.0,0.0
1988-06-24,0.0,0.0


Load met data from Daymet

In [10]:
nldi = NLDI()

# Get geometry and ensure CRS is correct
basin = nldi.get_basins(station_id) #get basin information, we could load the files that we saved too
geometry = basin.to_crs("EPSG:4326").geometry[0] # Get the bounding box of the geometry
start_date = "1980-10-01" # Start of water year, started with a smaller range
end_date = datetime.datetime.today().strftime('%Y-%m-%d') # End date is today, but could be set to the end of the water year
var = ["prcp", "tmin", "tmax",'srad', 'swe', 'vp', 'dayl'] # Variables to fetch, precip, temperature, solar radiation, snow water equivalent, vapor pressure, day length
dates = (start_date, end_date ) # Started with a smaller range to test

In [17]:
#Get geometry and ensure CRS is correct
basin = NLDI().get_basins(station_id)
geometry_centroid = geometry.centroid
centroid = (geometry_centroid.x, geometry_centroid.y)


#Fetch data - authentication now happens automatically via earthaccess/.netrc
# Try this simplified call first
met_df = daymet.get_bycoords(centroid, dates, variables=var)

met_df.head()

,dayl (s),prcp (mm/day),srad (W/m2),swe (kg/m2),tmax (degrees C),tmin (degrees C),vp (Pa)
time,,,,,,,
1980-01-01,32891.95,3.42,121.71,4.06,4.01,-2.30,515.93
1980-01-02,32934.47,0.00,222.03,4.06,2.88,-6.07,388.27
1980-01-03,32980.46,0.00,258.32,4.06,1.07,-9.95,287.01
1980-01-04,33029.90,0.00,257.22,3.11,7.46,-3.74,463.41
1980-01-05,33082.75,0.00,290.88,2.48,10.04,-4.51,437.04


In [18]:
# clean the dataframe, rename the columns
met_df.rename(columns={"prcp (mm/day)": "prcp_mm_day",'srad (W/m2)': "srad_W_m2", 'swe (kg/m2)': "swe_cm", "vp (Pa)": "vp_Pa", "dayl (s)": "dayl_s", "tmin (degrees C)": "tmin_C", "tmax (degrees C)": "tmax_C"}, inplace=True)
#Calculate Mean Temperature
met_df["tmean"] = (met_df.tmax_C + met_df.tmin_C) / 2
#convert swe from kg/m2 to cm, 1 kg/m2 is equivalent to 0.1 cm of water
met_df["swe_cm"] = met_df["swe_cm"] * 0.1

#set the index to name to date
met_df.index.name = "Date"

met_df.head()

,dayl_s,prcp_mm_day,srad_W_m2,swe_cm,tmax_C,tmin_C,vp_Pa,tmean
Date,,,,,,,,
1980-01-01,32891.95,3.42,121.71,0.406,4.01,-2.30,515.93,0.855
1980-01-02,32934.47,0.00,222.03,0.406,2.88,-6.07,388.27,-1.595
1980-01-03,32980.46,0.00,258.32,0.406,1.07,-9.95,287.01,-4.440
1980-01-04,33029.90,0.00,257.22,0.311,7.46,-3.74,463.41,1.860
1980-01-05,33082.75,0.00,290.88,0.248,10.04,-4.51,437.04,2.765


Load data from NLDS

In [21]:
# Get geometry and ensure CRS is correct
basin = nldi.get_basins(station_id) #get basin information, we could load the files that we saved too
geometry = basin.to_crs("EPSG:4326").geometry[0] # Get the bounding box of the geometry
basin_polygon_coords = list(geometry.exterior.coords)

In [22]:
#It crashes if we try to get too much data at once, so we will get daily data for a smaller range of dates
Daily_NLDAS_df1 = getData.get_NLDAS_daily(basin_polygon_coords, 
                                          begin_date='1980-01-01', 
                                          end_date='1990-01-1')
Daily_NLDAS_df1.head()

Authenticating with Earth Engine...
Initializing Earth Engine...
Earth Engine initialized successfully.


,convective_fraction,longwave_radiation,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v
Date,,,,,,,,,,,
1980-01-01,0.027078,282.673580,10.221365,0.014333,77863.835333,68.638128,0.003682,-1.440993,0.162109,2.175501,0.192600
1980-01-02,0.000000,262.000530,32.955574,0.018108,77964.978745,92.015744,0.003533,-3.024252,0.027528,1.859774,-0.657435
1980-01-03,0.000000,226.654711,0.000000,0.027583,77895.113782,92.887556,0.002868,-4.668699,0.000000,1.753648,2.879460
1980-01-04,0.000000,229.820631,0.000000,0.042707,77711.315911,108.177691,0.002619,-2.795418,0.000560,2.279286,3.405962
1980-01-05,0.000000,274.496061,0.000000,0.070334,77630.256996,87.816476,0.002290,-0.301378,0.013539,2.853367,4.511605


In [23]:
#It crashes if we try to get too much data at once, so we will get daily data for a smaller range of dates
Daily_NLDAS_df2 = getData.get_NLDAS_daily(basin_polygon_coords, 
                                          begin_date='1990-01-01', 
                                          end_date='2000-01-1')
Daily_NLDAS_df2.head()

Authenticating with Earth Engine...
Initializing Earth Engine...
Earth Engine initialized successfully.


,convective_fraction,longwave_radiation,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v
Date,,,,,,,,,,,
1990-01-01,0.000000,201.028472,0.000000,0.043603,77497.611508,105.191510,0.002177,-4.374669,0.000000,1.532501,4.097788
1990-01-02,0.162483,258.874941,42.215013,0.056446,76756.860853,95.911512,0.002972,-2.637200,0.344942,3.419418,3.702957
1990-01-03,0.018816,206.821447,6.091273,0.064650,77317.553027,115.823116,0.001961,-8.158093,0.003087,2.127582,-2.939185
1990-01-04,0.000000,174.579489,0.000000,0.032258,77961.230309,113.530631,0.001593,-11.051695,0.124079,1.932025,1.658583
1990-01-05,0.000000,243.548780,0.000000,0.033517,78024.046614,108.850804,0.002442,-6.987228,0.000072,2.534853,1.067739


In [24]:
#It crashes if we try to get too much data at once, so we will get daily data for a smaller range of dates
Daily_NLDAS_df3 = getData.get_NLDAS_daily(basin_polygon_coords, 
                                          begin_date='2000-01-01', 
                                          end_date='2010-01-1')
Daily_NLDAS_df3.head()

Authenticating with Earth Engine...
Initializing Earth Engine...
Earth Engine initialized successfully.


,convective_fraction,longwave_radiation,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v
Date,,,,,,,,,,,
2000-01-01,0.000000,209.801755,0.000000,0.023330,77023.533520,100.100026,0.002160,-6.283333,0.000507,2.123703,1.904920
2000-01-02,0.030476,225.059356,9.789795,0.030217,76937.197786,84.108382,0.002242,-7.981138,0.259631,3.012728,-0.073844
2000-01-03,0.000000,238.197302,5.027190,0.036265,77738.255868,81.666724,0.001946,-9.688405,0.187595,3.876129,-0.234470
2000-01-04,0.000000,214.394915,0.000000,0.030517,78238.315239,59.880062,0.001761,-10.104266,0.000000,2.057542,3.957667
2000-01-05,0.000000,259.958289,6.331623,0.033197,77800.864640,72.240537,0.002636,-6.247450,0.459437,4.991291,0.260574


In [25]:
#It crashes if we try to get too much data at once, so we will get daily data for a smaller range of dates
Daily_NLDAS_df4 = getData.get_NLDAS_daily(basin_polygon_coords, 
                                          begin_date='2010-01-01', 
                                          end_date='2020-01-1')
Daily_NLDAS_df4.head()

Authenticating with Earth Engine...
Initializing Earth Engine...
Earth Engine initialized successfully.


,convective_fraction,longwave_radiation,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v
Date,,,,,,,,,,,
2010-01-01,0.000000,269.489407,0.938634,0.041546,78103.035826,82.788554,0.002690,-5.179026,0.055088,2.289293,5.415286
2010-01-02,0.113519,273.303531,18.998767,0.017628,78006.996526,83.476381,0.003546,-2.897042,0.102350,3.746492,2.563209
2010-01-03,0.000000,209.143131,0.000000,0.006046,78137.554047,109.533837,0.002464,-7.010101,0.000998,1.652273,1.397331
2010-01-04,0.000000,196.569630,0.000000,0.012423,78194.509504,111.098744,0.002120,-7.788743,0.000000,0.117632,2.631311
2010-01-05,0.000000,246.154694,0.000000,0.020477,77924.180652,70.536153,0.002509,-5.509124,0.000000,1.927658,2.983824


In [26]:
#combine the four dataframes into one
Daily_NLDAS_df = pd.concat([Daily_NLDAS_df1, Daily_NLDAS_df2, Daily_NLDAS_df3, Daily_NLDAS_df4], ignore_index=False)

Daily_NLDAS_df.head()

,convective_fraction,longwave_radiation,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v
Date,,,,,,,,,,,
1980-01-01,0.027078,282.673580,10.221365,0.014333,77863.835333,68.638128,0.003682,-1.440993,0.162109,2.175501,0.192600
1980-01-02,0.000000,262.000530,32.955574,0.018108,77964.978745,92.015744,0.003533,-3.024252,0.027528,1.859774,-0.657435
1980-01-03,0.000000,226.654711,0.000000,0.027583,77895.113782,92.887556,0.002868,-4.668699,0.000000,1.753648,2.879460
1980-01-04,0.000000,229.820631,0.000000,0.042707,77711.315911,108.177691,0.002619,-2.795418,0.000560,2.279286,3.405962
1980-01-05,0.000000,274.496061,0.000000,0.070334,77630.256996,87.816476,0.002290,-0.301378,0.013539,2.853367,4.511605


Load streamflow data

In [27]:
#get streamflow information using the NWIS id for the gage and ghte get_usgs_streamflow function in getData.py 
streamflow = getData.get_usgs_streamflow(station_id)

Retrieving data for Site: 10133980 from 1980-01-01 to 2026-04-13...


In [28]:
streamflow.head()

,site_no,00060_Mean,00060_Mean_cd
datetime,,,
2007-07-06 00:00:00+00:00,10133980,8.30,"A, e"
2007-07-07 00:00:00+00:00,10133980,8.34,A
2007-07-08 00:00:00+00:00,10133980,8.52,A
2007-07-09 00:00:00+00:00,10133980,7.69,A
2007-07-10 00:00:00+00:00,10133980,9.55,A


In [30]:
streamflow_df = dataprocessing.clean_nwis_dataframe(streamflow)
#set the index name to Date
streamflow_df.index.name = "Date"
streamflow_df.head()

,site_no,flow_cfs
Date,,
2007-07-06,10133980,8.30
2007-07-07,10133980,8.34
2007-07-08,10133980,8.52
2007-07-09,10133980,7.69
2007-07-10,10133980,9.55


Now put all the data together

In [33]:
#find the latest start date and the earliest end date for SNOTEL_df, met_df, cleaned
begin_date = max([df.index.min() for df in [SNOTEL_df, met_df, streamflow_df, Daily_NLDAS_df]]) 
end_date = min([df.index.max() for df in [SNOTEL_df, met_df, streamflow_df, Daily_NLDAS_df]]) 

#clip each dataframe to have the same begin and end dates
SNOTEL_df = SNOTEL_df[(SNOTEL_df.index >= begin_date) & (SNOTEL_df.index <= end_date)]
met_df = met_df[(met_df.index >= begin_date) & (met_df.index <= end_date)]
streamflow_df = streamflow_df[(streamflow_df.index >= begin_date) & (streamflow_df.index <= end_date)]
Daily_NLDAS_df = Daily_NLDAS_df[(Daily_NLDAS_df.index >= begin_date) & (Daily_NLDAS_df.index <= end_date)]

In [34]:
#merge the SNOTEL_df, met_df, and streamflow dataframes
Hydro_df = pd.concat([SNOTEL_df, met_df, Daily_NLDAS_df,streamflow_df], axis=1)
#put the site_no column, second to last, and streamfow column, last column, as the first two columns in the dataframe
cols = Hydro_df.columns.tolist()
cols = cols[-2:] + cols[:-2]
Hydro_df = Hydro_df[cols]
Hydro_df.head()

,site_no,flow_cfs,814_SWE_cm,684_SWE_cm,dayl_s,prcp_mm_day,srad_W_m2,swe_cm,tmax_C,tmin_C,...,longwave_radiation,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v
Date,,,,,,,,,,,,,,,,,,,,,
2007-07-06,10133980,8.30,0.0,0.0,53344.21,0.0,482.56,0.0,32.21,13.37,...,302.415233,224.432105,0.395784,78416.761320,329.684526,0.006271,22.156583,0.0,-2.653197,1.484444
2007-07-07,10133980,8.34,0.0,0.0,53289.69,0.0,474.02,0.0,31.59,13.07,...,298.836368,106.540631,0.372804,78249.155624,308.000889,0.005430,21.975978,0.0,1.615503,3.137325
2007-07-08,10133980,8.52,0.0,0.0,53231.81,0.0,505.05,0.0,30.93,10.65,...,313.732067,148.169257,0.389178,78083.377693,332.230357,0.006165,21.141001,0.0,1.509399,-0.751796
2007-07-09,10133980,7.69,0.0,0.0,53170.63,0.0,503.75,0.0,30.64,10.56,...,279.122038,51.228500,0.378591,78110.992610,337.851225,0.004553,20.368652,0.0,1.709911,0.375726
2007-07-10,10133980,9.55,0.0,0.0,53106.18,0.0,516.91,0.0,31.84,10.87,...,276.366165,38.147736,0.380783,78210.982911,339.553340,0.004258,20.967940,0.0,-0.678148,-0.130966


In [35]:
#all of the NaN values here should be 0, fill them
Hydro_df = Hydro_df.fillna(0)
Hydro_df.head()

,site_no,flow_cfs,814_SWE_cm,684_SWE_cm,dayl_s,prcp_mm_day,srad_W_m2,swe_cm,tmax_C,tmin_C,...,longwave_radiation,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v
Date,,,,,,,,,,,,,,,,,,,,,
2007-07-06,10133980,8.30,0.0,0.0,53344.21,0.0,482.56,0.0,32.21,13.37,...,302.415233,224.432105,0.395784,78416.761320,329.684526,0.006271,22.156583,0.0,-2.653197,1.484444
2007-07-07,10133980,8.34,0.0,0.0,53289.69,0.0,474.02,0.0,31.59,13.07,...,298.836368,106.540631,0.372804,78249.155624,308.000889,0.005430,21.975978,0.0,1.615503,3.137325
2007-07-08,10133980,8.52,0.0,0.0,53231.81,0.0,505.05,0.0,30.93,10.65,...,313.732067,148.169257,0.389178,78083.377693,332.230357,0.006165,21.141001,0.0,1.509399,-0.751796
2007-07-09,10133980,7.69,0.0,0.0,53170.63,0.0,503.75,0.0,30.64,10.56,...,279.122038,51.228500,0.378591,78110.992610,337.851225,0.004553,20.368652,0.0,1.709911,0.375726
2007-07-10,10133980,9.55,0.0,0.0,53106.18,0.0,516.91,0.0,31.84,10.87,...,276.366165,38.147736,0.380783,78210.982911,339.553340,0.004258,20.967940,0.0,-0.678148,-0.130966


In [36]:
#take a look around peak SWE to make sure we have snotel values, early season can be tricky to assess
Hydro_df.loc['2019-03-01':'2019-04-01']

,site_no,flow_cfs,814_SWE_cm,684_SWE_cm,dayl_s,prcp_mm_day,srad_W_m2,swe_cm,tmax_C,tmin_C,...,longwave_radiation,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v
Date,,,,,,,,,,,,,,,,,,,,,
2019-03-01,10133980,29.3,59.182,41.402,39735.00,4.50,229.10,3.697,4.76,-2.36,...,253.130491,10.174874,0.041728,77498.839588,180.216409,0.003498,-0.978867,0.070439,2.467982,1.328314
2019-03-02,10133980,31.0,59.182,41.656,39896.12,10.13,202.01,3.564,3.13,-2.82,...,279.675708,0.000000,0.045722,77190.343313,135.680171,0.003342,-1.454088,0.208132,0.195621,0.615507
2019-03-03,10133980,30.1,60.452,42.926,40057.83,0.00,328.12,3.523,2.09,-5.03,...,270.243391,2.355131,0.031974,77332.816264,144.037102,0.003305,-3.562047,0.472308,0.985324,-0.680750
2019-03-04,10133980,28.9,61.468,43.180,40220.08,0.00,414.58,3.523,1.99,-7.73,...,186.457257,0.163441,0.044236,77577.387460,214.167326,0.002476,-6.743963,0.006640,0.535531,0.311450
2019-03-05,10133980,28.0,61.468,43.180,40382.86,0.00,460.18,3.504,6.48,-5.55,...,229.883986,0.000323,0.026713,77722.403472,191.493331,0.002796,-4.275926,0.001906,-0.469001,3.095125
2019-03-06,10133980,30.3,62.484,43.434,40546.12,9.45,226.78,3.265,6.34,-0.31,...,276.921744,0.320645,0.013515,77074.712996,144.271399,0.004002,0.334720,0.220339,-0.194004,4.556691
2019-03-07,10133980,40.5,64.770,44.450,40709.83,9.97,200.29,3.015,5.68,-0.04,...,273.178112,28.924921,0.009326,77053.532444,133.148101,0.004162,0.103837,0.622999,2.937256,4.582194
2019-03-08,10133980,39.1,66.294,45.974,40873.96,12.52,232.18,2.864,4.31,-2.40,...,266.288193,15.758067,0.037892,76724.665596,173.743739,0.003758,-0.969265,0.676422,0.219402,1.974572
2019-03-09,10133980,34.7,68.072,47.498,41038.49,4.33,287.65,3.297,1.76,-6.89,...,253.463826,15.741848,0.044062,76855.757717,177.113381,0.002843,-5.103026,0.333516,2.198983,1.167336


Save our data file!

In [38]:
#save data with station ID in the filename
#if HydroDF directory doesn't exist, create it
if not os.path.exists("files/HydroDF"):
    os.makedirs("files/HydroDF")
Hydro_df.to_csv(f"files/HydroDF/HydroDF_{basinname}_{station_id}.csv")